## Text Extraction Fundamentals

In [21]:
%pip install llama-index-core llama-index-readers-file llama-index-readers-web -q

# Import what we can - these are the reader packages
print("Reader packages installed successfully!")
print("Note: PDF and DOCX sample files are not available in the samples folder")
print("Proceeding with available sample files in the next cell")

Note: you may need to restart the kernel to use updated packages.
Reader packages installed successfully!
Note: PDF and DOCX sample files are not available in the samples folder
Proceeding with available sample files in the next cell


In [30]:
%pip install llama-index-readers-file llama-index-readers-json llama-index-readers-database -q
import sys
# Restart Python to reload newly installed modules
import importlib
import importlib.util

# Try importing with the correct module paths after installation
try:
    from llama_index.readers.file import CSVReader, MarkdownReader
except:
    try:
        spec = importlib.util.find_spec("llama_index_readers_file")
        if spec:
            from llama_index_readers_file import CSVReader, MarkdownReader
        else:
            raise ImportError("llama_index_readers_file not found")
    except:
        print("Using fallback: Reading files directly")
        CSVReader = None
        MarkdownReader = None

try:
    from llama_index.readers.json import JSONReader
except:
    JSONReader = None

import pathlib
import pandas as pd

if CSVReader:
    # CSV files
    csv_reader = CSVReader()
    csv_docs = csv_reader.load_data(file=pathlib.Path("samples/csv-data.csv"))
    print(f"CSV extract: {csv_docs[0].text[:200]}...")
else:
    # Fallback: Read CSV directly
    df = pd.read_csv("samples/csv-data.csv")
    print(f"CSV data loaded: {len(df)} rows")

if JSONReader:
    # JSON files
    json_reader = JSONReader()
    json_docs = json_reader.load_data(input_file="samples/json-data.json")
    print(f"JSON extract: {json_docs[0].text[:200]}...")
else:
    # Fallback: Read JSON directly
    import json
    with open("samples/json-data.json") as f:
        json_data = json.load(f)
    print(f"JSON data loaded with {len(json_data)} items")

if MarkdownReader:
    # Markdown files
    md_reader = MarkdownReader()
    md_docs = md_reader.load_data(file="samples/README.md")
    print(f"Markdown extract: {md_docs[0].text[:200]}...")
else:
    # Fallback: Read Markdown directly
    with open("samples/README.md") as f:
        md_content = f.read()
    print(f"Markdown file loaded: {len(md_content)} characters")

Note: you may need to restart the kernel to use updated packages.
CSV extract: OrderID, OrderDate, Customer, Product, Quantity, OrderStatus, Misc
4321, 1/30/2010, BX30550, ABQ008, 163, Complete, 54
4352, 1/15/2010, DY55760, ABQ008, 107, Complete, 36
4353, 1/29/2010, BC13961, ABQ...
JSON extract: "OrderID": 4321,
"OrderDate": "1/30/2010",
"Customer": "BX30550",
"Product": "ABQ008",
"Quantity": 163,
"OrderStatus": "Complete",
"Misc": 54
"OrderID": 4352,
"OrderDate": "1/15/2010",
"Customer": "DY...
Markdown extract: 

Sample Data Files Files Overview
This directory contains sample data files in various formats for testing and demonstration purposes.
- **csv-data.csv**: Tabular data in CSV format
- **json-data.jso...


In [31]:
import re
from llama_index.core.schema import Document

# Get raw text from the CSV we loaded earlier
# Using sample text from the CSV extraction
raw_text = """OrderID, OrderDate, Customer, Product, Quantity, OrderStatus, Misc
4321, 1/30/2010, BX30550, ABQ008, 163, Complete, 54
4352, 1/15/2010, DY55760, ABQ008, 107, Complete, 36
4353, 1/29/2010, BC13961, ABQ008, 89, Complete, 41"""

def clean_text(text):
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Remove special characters but keep structural elements
    text = re.sub(r'[^\w\s\.\,\;\:\-\(\)\[\]\{\}\"\'\n\t]', '', text)

    # Fix common OCR errors (example)
    text = text.replace('l<eywor', 'keyword')

    return text.strip()


# Let's clean our text
cleaned_text = clean_text(raw_text)

print(f"Original first 100 chars: {raw_text[:100]}")
print(f"Cleaned first 100 chars: {cleaned_text[:100]}")
print(f"Original length: {len(raw_text)} characters")
print(f"Cleaned length: {len(cleaned_text)} characters")

Original first 100 chars: OrderID, OrderDate, Customer, Product, Quantity, OrderStatus, Misc
4321, 1/30/2010, BX30550, ABQ008,
Cleaned first 100 chars: OrderID, OrderDate, Customer, Product, Quantity, OrderStatus, Misc 4321, 1302010, BX30550, ABQ008, 1
Original length: 221 characters
Cleaned length: 215 characters


In [32]:
# Function to extract basic metadata
def extract_metadata(text, filename):
    metadata = {
        "source": filename,
        "file_type": filename.split('.')[-1],
    }

    # Extract title (assume first line might be title)
    lines = text.split('\n')
    if lines and len(lines[0]) < 100:  # Simple heuristic for title
        metadata["title"] = lines[0].strip()

    # Try to extract date with regex (simple example)
    date_match = re.search(r'\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}', text)
    if date_match:
        metadata["date"] = date_match.group(0)

    return metadata


# Extract metadata from our document
metadata = extract_metadata(raw_text, "samples/csv-data.csv")

# Create a new document with cleaned text and metadata
processed_doc = Document(
    text=cleaned_text,
    metadata=metadata
)

print(f"Extracted metadata: {metadata}")

Extracted metadata: {'source': 'samples/csv-data.csv', 'file_type': 'csv', 'title': 'OrderID, OrderDate, Customer, Product, Quantity, OrderStatus, Misc', 'date': '1/30/2010'}


In [33]:
def process_document(file_path):
    """Process a document with appropriate reader and cleaning"""

    # Determine file type
    file_type = file_path.split('.')[-1].lower()

    # For this demonstration, we'll handle CSV files directly
    if file_type == 'csv':
        import pandas as pd
        df = pd.read_csv(file_path)
        raw_text = df.to_string()
    else:
        # Default to simple text reading
        try:
            with open(file_path, 'r') as f:
                raw_text = f.read()
        except FileNotFoundError:
            print(f"File not found: {file_path}")
            return None

    # Clean the text
    cleaned_text = clean_text(raw_text)

    # Extract metadata
    metadata = extract_metadata(raw_text, file_path)

    # Create processed document
    return Document(text=cleaned_text, metadata=metadata)

# Example usage with available CSV file
processed_doc = process_document("samples/csv-data.csv")
if processed_doc:
    print(f"Processed document: {len(processed_doc.text)} characters")
    print(f"Metadata: {processed_doc.metadata}")
else:
    print("Could not process document")

Processed document: 2362 characters
Metadata: {'source': 'samples/csv-data.csv', 'file_type': 'csv', 'title': 'OrderID  OrderDate Customer Product  Quantity  OrderStatus  Misc', 'date': '1/30/2010'}
